# Chroma CRUD Operations #

### this notebook walks through create , read , update and delete operations with a local chroma vector store ###

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

## 1.Setup paths and the Vector Store ##

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Shivam Singh/advanced-rag-loaders/04_vector_stores')

In [4]:
# Load environment variables from the local .env file.

dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("GOOGLE_API_KEY"):
    raise ValueError("Please add your GOOGLE_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\.env


In [5]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo
Persist directory: c:\Users\Shivam Singh\advanced-rag-loaders\04_vector_stores\notebooks\chroma_langchain_db


In [6]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [19]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [8]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [10]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [11]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [12]:
print(uuid4())

de40f1ec-2f81-4f5e-8af4-2ff1098342d6


In [13]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=e34a0406-678a-4c32-abdd-4f794bc02a3e
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=ff4a83c8-538a-4c92-8cd9-8982f3e366d4
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=f8c3409d-4445-4b6a-bc7c-f2ce57a7229a
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=7345df51-dc72-46aa-89d3-1ff14289105a
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=da20cd2e-df63-4f87-8b95-9752cde17df6
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=8a9ed068-8a37-4b50-b8b4-63eb991a77ab
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [21]:
documents[0].id

'e34a0406-678a-4c32-abdd-4f794bc02a3e'

In [20]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
e34a0406-678a-4c32-abdd-4f794bc02a3e
ff4a83c8-538a-4c92-8cd9-8982f3e366d4
f8c3409d-4445-4b6a-bc7c-f2ce57a7229a
7345df51-dc72-46aa-89d3-1ff14289105a
da20cd2e-df63-4f87-8b95-9752cde17df6
8a9ed068-8a37-4b50-b8b4-63eb991a77ab
978d61bc-463e-42c7-a904-cdf27e0388b8
223ba861-f9af-4f4f-aa83-4ee3cc314826
8ed92884-f1a8-4a16-a091-1fe29c6e8e42
2f63391a-bf7d-427d-82bc-bb5fc7533726

Total inserted documents: 10


## 4. Read the Stored Data Back

In [25]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [36]:
print(raw_records)

{'ids': ['e34a0406-678a-4c32-abdd-4f794bc02a3e', 'ff4a83c8-538a-4c92-8cd9-8982f3e366d4', 'f8c3409d-4445-4b6a-bc7c-f2ce57a7229a', '7345df51-dc72-46aa-89d3-1ff14289105a', 'da20cd2e-df63-4f87-8b95-9752cde17df6', '8a9ed068-8a37-4b50-b8b4-63eb991a77ab', '978d61bc-463e-42c7-a904-cdf27e0388b8', '223ba861-f9af-4f4f-aa83-4ee3cc314826', '8ed92884-f1a8-4a16-a091-1fe29c6e8e42', '2f63391a-bf7d-427d-82bc-bb5fc7533726'], 'embeddings': array([[-0.01103104,  0.00782148,  0.00854845, ..., -0.0034849 ,
         0.00138828,  0.00087448],
       [-0.00921574,  0.0208057 ,  0.0177272 , ..., -0.00572145,
         0.00503731, -0.00229997],
       [-0.01825958,  0.00664425,  0.00223987, ...,  0.00163793,
         0.00960462,  0.01627775],
       ...,
       [-0.01938161,  0.01245969,  0.01450677, ..., -0.01006061,
         0.00820549,  0.02444995],
       [-0.00221321,  0.01343836, -0.00237054, ..., -0.01373989,
        -0.00628342,  0.00090962],
       [ 0.00235443,  0.00275899,  0.00821233, ..., -0.0111523 ,

In [31]:
print(raw_records["embeddings"][0:2,0:20].shape)

(2, 20)


In [32]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
e34a0406-678a-4c32-abdd-4f794bc02a3e
ff4a83c8-538a-4c92-8cd9-8982f3e366d4
f8c3409d-4445-4b6a-bc7c-f2ce57a7229a


In [33]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['223ba861-f9af-4f4f-aa83-4ee3cc314826',
 '8ed92884-f1a8-4a16-a091-1fe29c6e8e42',
 '2f63391a-bf7d-427d-82bc-bb5fc7533726']

In [34]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=223ba861-f9af-4f4f-aa83-4ee3cc314826
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=8ed92884-f1a8-4a16-a091-1fe29c6e8e42
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=2f63391a-bf7d-427d-82bc-bb5fc7533726
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [35]:
print(selected_documents)

[Document(id='223ba861-f9af-4f4f-aa83-4ee3cc314826', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='8ed92884-f1a8-4a16-a091-1fe29c6e8e42', metadata={'topic': 'Cricket', 'doc_number': 9}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='2f63391a-bf7d-427d-82bc-bb5fc7533726', metadata={'doc_number': 10, 'topic': 'Cricket'}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [37]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [46]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=7345df51-dc72-46aa-89d3-1ff14289105a
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=da20cd2e-df63-4f87-8b95-9752cde17df6
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
3. id=8a9ed068-8a37-4b50-b8b4-63eb991a77ab
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they make semantic search over embedded documents possible.



In [39]:
print(search_results)

[Document(id='7345df51-dc72-46aa-89d3-1ff14289105a', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'), Document(id='da20cd2e-df63-4f87-8b95-9752cde17df6', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'), Document(id='8a9ed068-8a37-4b50-b8b4-63eb991a77ab', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.')]


In [40]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='7345df51-dc72-46aa-89d3-1ff14289105a', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.349837988615036),
 (Document(id='da20cd2e-df63-4f87-8b95-9752cde17df6', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  0.45079097151756287),
 (Document(id='8a9ed068-8a37-4b50-b8b4-63eb991a77ab', metadata={'topic': 'RAG', 'doc_number': 6}, page_content='Vector stores are important in RAG because they make semantic search over embedded documents possible.'),
  0.49773699045181274),
 (Document(id='978d61bc-463e-42c7-a904-cdf27e0388b8', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  0.5911723375320435)]

## 6. Update Existing Documents

In [41]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['7345df51-dc72-46aa-89d3-1ff14289105a',
 '223ba861-f9af-4f4f-aa83-4ee3cc314826']

In [42]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=7345df51-dc72-46aa-89d3-1ff14289105a
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=223ba861-f9af-4f4f-aa83-4ee3cc314826
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [43]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
7345df51-dc72-46aa-89d3-1ff14289105a
223ba861-f9af-4f4f-aa83-4ee3cc314826


In [44]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=7345df51-dc72-46aa-89d3-1ff14289105a
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=223ba861-f9af-4f4f-aa83-4ee3cc314826
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [47]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [50]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=7345df51-dc72-46aa-89d3-1ff14289105a
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=da20cd2e-df63-4f87-8b95-9752cde17df6
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [51]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['8ed92884-f1a8-4a16-a091-1fe29c6e8e42',
 '2f63391a-bf7d-427d-82bc-bb5fc7533726']

In [52]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
8ed92884-f1a8-4a16-a091-1fe29c6e8e42
2f63391a-bf7d-427d-82bc-bb5fc7533726


In [53]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
e34a0406-678a-4c32-abdd-4f794bc02a3e
ff4a83c8-538a-4c92-8cd9-8982f3e366d4
f8c3409d-4445-4b6a-bc7c-f2ce57a7229a
7345df51-dc72-46aa-89d3-1ff14289105a
da20cd2e-df63-4f87-8b95-9752cde17df6
8a9ed068-8a37-4b50-b8b4-63eb991a77ab
978d61bc-463e-42c7-a904-cdf27e0388b8
223ba861-f9af-4f4f-aa83-4ee3cc314826

Deleted ids still present?
8ed92884-f1a8-4a16-a091-1fe29c6e8e42: False
2f63391a-bf7d-427d-82bc-bb5fc7533726: False


In [54]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
